<a href="https://colab.research.google.com/github/omegatron01/corrosion-rag-project/blob/main/notebooks/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Metal Corrosion Assessment System

An AI pipeline that detects corrosion from metal surface images and generates
fitness-for-service recommendations based on international engineering standards.

## What this demo does
1. Loads a pretrained MobileNetV2 vision model from GitHub
2. Builds a FAISS knowledge base from ISO, ASTM and NACE standards
3. Loads Gemma as the reasoning LLM
4. Uses MCP-style tool routing to search multiple sources before answering
5. Launches a Gradio interface for live image assessment

## Before you start
- Make sure you are on a T4 GPU runtime
- You will need a free Hugging Face account with Gemma access

## Hugging Face Setup (do this before running Cell 6)
1. Create a free account at huggingface.co
2. Accept Gemma terms at: https://huggingface.co/google/gemma-4-E4B-it
3. Create a token at: https://huggingface.co/settings/tokens
   (New token → any name → Read role → Generate → copy it)

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected!")
    print("Go to Runtime → Change runtime type → T4 GPU → Save")
    print("Then restart and run again.")

In [ ]:
# Clone project from GitHub and install all dependencies
# This gets the pretrained model, knowledge base, and source files

!git clone https://github.com/omegatron01/corrosion-rag-project.git
%cd corrosion-rag-project

print("\nProject cloned! Installing packages...")

!pip install torch torchvision -q
!pip install sentence-transformers faiss-cpu -q
!pip install transformers accelerate bitsandbytes -q
!pip install gradio -q

print("\nAll packages installed!")
print("\nProject structure:")
!find . -type f -not -path './.git/*' | sort

In [ ]:
# Build the FAISS knowledge base from standards documents
# Reads all .txt files in knowledge_base/
# Chunks them, embeds them, stores in FAISS

import subprocess

print("Building knowledge base from standards documents...")
print("="*50)

result = subprocess.run(
    ["python", "src/ingest.py"],
    capture_output=True,
    text=True)

print(result.stdout)

if result.returncode != 0:
    print("ERROR:")
    print(result.stderr)
else:
    print("="*50)
    print("Knowledge base ready!")

## Hugging Face Login Required

Gemma 4 is a gated model. You need to accept Google's terms before downloading.

Complete these steps before running the cell below:

Step 1 — Create a free account at huggingface.co if you don't have one

Step 2 — Accept Gemma 4 terms at:
https://huggingface.co/google/gemma-4-E4B-it
(Click "Agree and access repository")

Step 3 — Create an access token at:
https://huggingface.co/settings/tokens
(New token → any name → Read role → Generate → copy it)

Step 4 — Run the cell below and paste your token when prompted

In [ ]:
# Log into Hugging Face
# Your token is never stored or shared

from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Load all three models
# 1. MobileNetV2 vision model (from GitHub)
# 2. FAISS knowledge base + embedding model
# 3. Gemma 4 E4B (from Hugging Face)
# This cell takes 5-10 minutes on first run

import torch
import torch.nn.functional as F
from torchvision import transforms, models
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle
from PIL import Image
import os
import sys

sys.path.append("src")

base_dir          = "/content/corrosion-rag-project"
model_path        = os.path.join(base_dir, "model", "best_corrosion_model.pth")
faiss_index_path  = os.path.join(base_dir, "vector_store", "faiss_index.bin")
chunks_path       = os.path.join(base_dir, "vector_store", "chunks.pkl")
gemma_model_id    = "google/gemma-4-E4B-it"

class_names = ["CORROSION", "NO_CORROSION"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])])

print("Step 1: Loading vision model...")
corrosion_model = models.mobilenet_v2(weights=None)
corrosion_model.classifier[1] = nn.Linear(corrosion_model.classifier[1].in_features, 2)
corrosion_model.load_state_dict(torch.load(model_path, map_location=device))
corrosion_model.eval()
corrosion_model = corrosion_model.to(device)
print("Vision model loaded!")

print("\nStep 2: Loading knowledge base...")
faiss_index  = faiss.read_index(faiss_index_path)
with open(chunks_path, "rb") as f:
    data = pickle.load(f)
all_chunks    = data["chunks"]
all_metadata  = data["metadata"]
embed_model   = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Knowledge base loaded: {faiss_index.ntotal} chunks")

print("\nStep 3: Loading Gemma 4 E4B...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(gemma_model_id)
gemma_model = AutoModelForCausalLM.from_pretrained(
    gemma_model_id,
    quantization_config=bnb_config,
    device_map="auto")
print("Gemma 4 loaded!")

print("\n" + "="*50)
print("All models loaded. Ready to run the demo!")
print("="*50)

In [ ]:
# Define all pipeline functions
import urllib.request
import urllib.parse
import json
import tempfile
import gradio as gr
source_name_map = {
    "iso_8501_rust_grades.txt"   : "ISO 8501-1 Rust Grade Definitions",
    "astm_b117_acceptance.txt"   : "ASTM B117 Salt Spray Testing Standard",
    "nace_sp0169_pipelines.txt"  : "NACE SP0169 Pipeline Corrosion Control",
    "remediation_guidelines.txt" : "Corrosion Remediation Guidelines"}

def predict_corrosion(image):
    """
    Takes a PIL Image.
    Runs through MobileNetV2.
    Returns structured dict with grade and confidence.
    """
    tensor = val_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs       = corrosion_model(tensor)
        probabilities = F.softmax(outputs[0], dim=0)
        confidence, predicted_idx = torch.max(probabilities, dim=0)

    predicted_class  = class_names[predicted_idx.item()]
    confidence_score = confidence.item()

    if predicted_class.lower() == "corrosion":
        if confidence_score >= 0.90:
            grade = "Grade C"
        elif confidence_score >= 0.70:
            grade = "Grade B"
        else:
            grade = "Grade A"
    else:
        grade = "No Corrosion Detected"

    return {
        "corrosion_grade"   : grade,
        "corrosion_type"    : "surface corrosion",
        "confidence"        : confidence_score,
        "raw_prediction"    : predicted_class,
        "all_probabilities" : {
            class_names[i]: round(probabilities[i].item(), 4)
            for i in range(len(class_names))}}



In [ ]:
def search_knowledge_base(vision_result, k=3):
    """
    Searches local FAISS index for relevant standard chunks.
    Returns chunks and their professional source names.
    """
    query = (
        f"Standards for {vision_result['corrosion_grade']} "
        f"{vision_result['corrosion_type']} on structural metal. "
        f"Fitness for service assessment and recommended action."
    )

    query_vector = embed_model.encode([query]).astype("float32")
    faiss.normalize_L2(query_vector)
    distances, indices = faiss_index.search(query_vector, k=k)

    chunks  = [all_chunks[i]             for i in indices[0]]
    sources = [all_metadata[i]["source"] for i in indices[0]]

    # Map to professional names, remove duplicates
    seen = []
    for s in sources:
        name = source_name_map.get(s, s)
        if name not in seen:
            seen.append(name)

    return chunks, seen

In [ ]:
#Wikipedia search
def search_wikipedia(query):
    """
    Searches Wikipedia using their free REST API.
    No API key needed.
    Returns plain text summary or empty string if unavailable.
    """
    try:
        encoded = urllib.parse.quote(query)
        url     = f"https://en.wikipedia.org/api/rest_v1/page/summary/{encoded}"
        req     = urllib.request.Request(
            url,
            headers={"User-Agent": "CorrosionRAG/1.0"}
        )
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read())

        if "extract" in data and data["extract"]:
            return {
                "success" : True,
                "title"   : data.get("title", query),
                "content" : data["extract"][:800]
            }
        return {"success": False}

    except Exception:
        return {"success": False}

In [ ]:
# Tool Router MCP-style orchestration
def run_tools(vision_result):
    """
    Runs all available tools for the given vision result.
    Combines their output into a single context string for Gemma.
    Returns (combined_context, tools_used).
    """
    combined_context = ""
    tools_used       = []

    # Tool 1 — FAISS knowledge base
    print("  Running Tool 1: Local standards database...")
    chunks, sources = search_knowledge_base(vision_result)
    tools_used.extend(sources)
    combined_context += "\n[LOCAL STANDARDS DATABASE]\n"
    for chunk in chunks:
        combined_context += f"{chunk}\n"

    # Tool 2 — Wikipedia
    print("  Running Tool 2: Wikipedia search...")
    wiki_query  = f"ISO 8501 {vision_result['corrosion_grade']} rust corrosion steel"
    wiki_result = search_wikipedia(wiki_query)

    if wiki_result["success"]:
        tools_used.append(f"Wikipedia: {wiki_result['title']}")
        combined_context += f"\n[WIKIPEDIA: {wiki_result['title']}]\n"
        combined_context += wiki_result["content"] + "\n"
        print(f"  Wikipedia found: {wiki_result['title']}")
    else:
        print("  Wikipedia unavailable, continuing without it")

    return combined_context, tools_used


In [ ]:
# Gemma Reasoning
def ask_gemma(vision_result, combined_context):
    """
    Sends vision result and combined tool context to Gemma 4.
    Returns plain text assessment.
    """
    prompt = f"""<start_of_turn>user
You are a corrosion engineering expert assistant.
Do not use any Markdown formatting, asterisks, or bold text.
Write in plain text only.

You have been given information from multiple sources including
a local standards database and Wikipedia.
Use all of it to assess whether this metal component is fit for service.

Always:
- State clearly if the component is fit for service or not
- Cite which source or standard supports your answer
- Give a specific recommended action
- Be concise and professional

VISION MODEL DETECTION:
- Corrosion Grade: {vision_result['corrosion_grade']}
- Corrosion Type:  {vision_result['corrosion_type']}
- Confidence:      {vision_result['confidence'] * 100:.0f}%

INFORMATION FROM TOOLS:
{combined_context}

What is your assessment and recommendation?
<end_of_turn>
<start_of_turn>model
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(gemma_model.device)

    with torch.no_grad():
        output = gemma_model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    full_output = tokenizer.decode(output[0], skip_special_tokens=True)
    answer      = full_output.split("model")[-1].strip()
    return answer

In [ ]:
def assess_metal_image(image):
    """
    Called by Gradio when user clicks Assess Corrosion.
    Runs the full pipeline and returns formatted results.
    """
    print("\nStarting assessment pipeline...")

    # Stage 1: Vision model
    print("Stage 1: Running vision model...")
    vision_result = predict_corrosion(image)
    print(f"  Detected: {vision_result['corrosion_grade']} "
          f"({vision_result['confidence']*100:.1f}% confidence)")

    # Format probabilities
    probs     = vision_result["all_probabilities"]
    prob_text = "\n".join([
        f"  {k.replace('_', ' ').title()}: {v*100:.1f}%"
        for k, v in probs.items()
    ])

    vision_output = f"""VISION MODEL RESULT
-------------------
Prediction: {vision_result['raw_prediction'].replace('_', ' ').title()}
Corrosion Grade: {vision_result['corrosion_grade']}
Confidence: {vision_result['confidence']*100:.1f}%

Probabilities:
{prob_text}
"""

    # Stage 2: Tool routing (MCP-style)
    print("Stage 2: Running tool router...")
    combined_context, tools_used = run_tools(vision_result)

    # Stage 3: Gemma assessment
    print("Stage 3: Running Gemma 4...")
    answer = ask_gemma(vision_result, combined_context)

    # Format sources — remove duplicates
    seen = []
    for t in tools_used:
        if t not in seen:
            seen.append(t)
    standards = "\n".join([f"  - {name}" for name in seen])

    gemma_output = f"""GEMMA'S ASSESSMENT
------------------
{answer}

SOURCES USED:
{standards}
"""

    print("Assessment complete!")
    return vision_output, gemma_output


print("All pipeline functions defined and ready!")

In [ ]:
# Launch Gradio interface
import gradio as gr

with gr.Blocks(title="Metal Corrosion Assessment System") as app:

    gr.Markdown("# Metal Corrosion Assessment System")
    gr.Markdown(
        "Upload a photo of a metal surface to receive an automated "
        "corrosion grade and fitness-for-service recommendation "
        "based on international engineering standards.")
    gr.Markdown(
        "_Powered by MobileNetV2 · FAISS · Wikipedia · Gemma 4 E4B_")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload Metal Image")
            submit_btn = gr.Button(
                "Assess Corrosion",
                variant="primary")
            gr.Markdown(
                "**Supported formats:** JPG, JPEG, PNG")

        with gr.Column():
            vision_output = gr.Textbox(
                label="Vision Model Result",
                lines=10,
                show_copy_button=True)
            gemma_output = gr.Textbox(
                label="Gemma Assessment",
                lines=18,
                show_copy_button=True)

    submit_btn.click(
        fn=assess_metal_image,
        inputs=image_input,
        outputs=[vision_output, gemma_output])

    gr.Markdown(
        "---\n"
        "**GitHub:** https://github.com/omegatron01/corrosion-rag-project")

app.launch(share=True)